In [ ]:
!pip -q install evaluate rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [2]:
import random
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate

*seed*

In [3]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

### Load SAMSum dataset

In [4]:
samsum = load_dataset("knkarthick/samsum")
print(samsum)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


_Make smaller splits for easier fine-tuning_

In [5]:
train_raw = samsum["train"].shuffle(seed=42).select(range(4000))
val_raw = samsum["validation"].shuffle(seed=42).select(range(500))
test_raw = samsum["test"].shuffle(seed=42).select(range(500))

print(len(train_raw), len(val_raw), len(test_raw))

4000 500 500


_Show one raw sample_

In [6]:
sample = train_raw[0]

print("Dialogue:\n")
print(sample["dialogue"])
print("\n" + "=" * 80 + "\n")
print("Reference Summary:\n")
print(sample["summary"])

Dialogue:

Adam: <file_video>
Adam: what do you think
Hector: give me a sec
Hector: ok watching
Adam: let me know
Hector: can't really hear a lot there ;/
Adam: yeah ;/
Adam: i think i need to record it somehow else
Adam: maybe through the interface and software
Hector: that definitely is a great idea!
Hector: i guess that's why i gave you the interface and installed it :D
Adam: yeah xd
Adam: ok i'll try to figure it out later
Hector: ok
Hector: i'll be waiting :P


Reference Summary:

Adam will record it somewhere else through the interface and software. Hector gave and installed the interface before.


### Load tokenizer and model

In [7]:
MODEL_NAME = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

cuda


_Text settings_

In [8]:
PREFIX = "summarize: "
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 64
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-4

### See how tokenizer works

In [9]:
input_text = PREFIX + sample["dialogue"]
target_text = sample["summary"]

input_tokens = tokenizer(input_text, max_length=MAX_INPUT_LENGTH, truncation=True)
target_tokens = tokenizer(text_target=target_text, max_length=MAX_TARGET_LENGTH, truncation=True)

print("Input token count :", len(input_tokens["input_ids"]))
print("Target token count:", len(target_tokens["input_ids"]))
print()
print("Input ids preview :", input_tokens["input_ids"][:30])
print("Target ids preview:", target_tokens["input_ids"][:30])

Input token count : 148
Target token count: 22

Input ids preview : [21603, 10, 7124, 10, 3, 2, 11966, 834, 17241, 3155, 7124, 10, 125, 103, 25, 317, 216, 5317, 10, 428, 140, 3, 9, 4220, 216, 5317, 10, 3, 1825, 3355]
Target ids preview: [7124, 56, 1368, 34, 5775, 1307, 190, 8, 3459, 11, 889, 5, 216, 5317, 1891, 11, 3029, 8, 3459, 274, 5, 1]


### Custom dataset class

In [10]:
class SamsumSummaryDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_input_length, max_target_length, prefix):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length
        self.prefix = prefix
        
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, index):
        item = self.dataset[index]
        
        input_text = self.prefix + item["dialogue"]
        target_text = item["summary"]
        
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            truncation=True,
            return_tensors="pt"
        )
        
        target_encoding = self.tokenizer(
            text_target=target_text,
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids" : input_encoding["input_ids"].squeeze(0),
            "attention_mask" : input_encoding["attention_mask"].squeeze(0),
            "labels" : target_encoding["input_ids"].squeeze(0)
        }

_Build train, validation, and test datasets_

In [11]:
train_dataset = SamsumSummaryDataset(train_raw, tokenizer, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH, PREFIX)
val_dataset = SamsumSummaryDataset(val_raw, tokenizer, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH, PREFIX)
test_dataset = SamsumSummaryDataset(test_raw, tokenizer, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH, PREFIX)

print(len(train_dataset), len(val_dataset), len(test_dataset))

4000 500 500


_Custom collate function_

In [12]:
def collate_fn(batch):
    input_ids = [x["input_ids"] for x in batch]
    attention_mask = [x["attention_mask"] for x in batch]
    labels = [x["labels"] for x in batch]
    
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=tokenizer.pad_token_id)
    
    labels[labels == tokenizer.pad_token_id] = -100
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

_DataLoaders_

In [13]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

_Inspect one batch_

In [14]:
batch = next(iter(train_loader))

print("input_ids shape     :", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels shape        :", batch["labels"].shape)
print()
print("First input ids     :", batch["input_ids"][0][:25])
print("First labels ids    :", batch["labels"][0][:25])

input_ids shape     : torch.Size([8, 256])
attention_mask shape: torch.Size([8, 256])
labels shape        : torch.Size([8, 53])

First input ids     : tensor([21603,    10,  5104,    10,  9459,    62,    31,    60,   352,    12,
        12263,    48,  1248,    58, 23499,   159,    10,  7271,     6,  1907,
           51,     9,    19,  6802, 18811])
First labels ids    : tensor([23499,   159,    11,  5104,    56,   281,    12, 12263,    48,  1248,
            5,    37,  1907,    51,     9,    19,  6802,     5,  5104,    56,
          960,    21,  7534,    28,    71])


_Decode one batch example_

In [15]:
input_ids = batch["input_ids"][0]
label_ids = batch["labels"][0]

clean_label_ids = torch.where(label_ids == -100, tokenizer.pad_token_id, label_ids)

decoded_input = tokenizer.decode(input_ids, skip_special_tokens=False)
decoded_label = tokenizer.decode(clean_label_ids, skip_special_tokens=False)

print("Decoded Input:\n")
print(decoded_input[:800])
print("\n" + "=" * 80 + "\n")
print("Decoded Label:\n")
print(decoded_label)

Decoded Input:

summarize: Alex: Hey we're going to Greece this summer? Sakis: Ye, grandma is sick nowadays Alex: I know Alex: Should I search tix for us? Sakis: Ye its a good time to book em now Alex: What airline u prefer? Sakis: Aegean? Alex: Sure</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><p


Decoded Label:

Sakis and Alex will go to Greece this summer. The grandma is sick. Alex will search for flights with Aegean Airlines.</s><pad><pad><pad><pad><pad><pad><pad><pad><pad

### Load ROUGE metric

In [18]:
rouge = evaluate.load("rouge")

### One forward pass

In [19]:
small_batch = next(iter(train_loader))
small_batch = {k: v.to(device) for k, v in small_batch.items()}

outputs = model(
    input_ids=small_batch["input_ids"],
    attention_mask=small_batch["attention_mask"],
    labels=small_batch["labels"]
)

print("Loss        :", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

Loss        : 2.347338914871216
Logits shape: torch.Size([8, 32, 32128])


### Train loop

In [20]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_size = 0
    
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        
    return total_loss / len(dataloader)